# Milestone 4 - Sequential Models & CRNNs (CNN + RNN/LSTM/GRU)

**Student Email:** 24f2007692@ds.study.iitm.ac.in

This notebook uses helper functions from the [milestone-4-codebase](https://github.com/Photon-08/milestone-4-codebase/tree/main) repository and the competition dataset to answer all 6 questions.

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import numpy as np
import random
import os
import glob
import math
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

In [ ]:
def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

## Paths

In [ ]:
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

---
## Helper Function: generate_synthetic_dataset

In [ ]:
def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration

    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)

    for genre in genres:
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)

        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders:
            continue

        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)

                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)

            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)

                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)

                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]

                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)

                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)

                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

In [ ]:
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)

---
## Q1
How many .wav files should exist in /kaggle/working/synthetic_mashups/train/?

In [ ]:
samples_per_genre = 50
num_genres = 10

total_wav_files = samples_per_genre * num_genres
wav_files = glob.glob(os.path.join(OUTPUT_PATH, '**', '*.wav'), recursive=True)

q1_output = {
    "expected_total_wav_files": int(total_wav_files),
    "files_found_on_disk": int(len(wav_files))
}
q1_output

Q1 Answer: 500

---
## Q2
What is the exact tensor shape of the resulting waveform?

In [ ]:
target_sr = 22050
duration = 30

expected_samples = target_sr * duration
sample_wav = glob.glob(os.path.join(OUTPUT_PATH, '**', '*.wav'), recursive=True)[0]
waveform, sr = torchaudio.load(sample_wav)

q2_output = {
    "expected_samples": int(expected_samples),
    "actual_waveform_shape": tuple(waveform.shape),
    "sample_rate": int(sr)
}
q2_output

Q2 Answer: (2, 661500)

---
## Helper Function: extract_and_save_features

In [ ]:
def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)

    if not wav_files:
        return

    for wav_path in wav_files:
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')

        out_path.parent.mkdir(parents=True, exist_ok=True)

        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)

        torch.save(mel_spec_db, out_path)

In [ ]:
INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

---
## Q3
What is the exact dimension shape of the pre-computed .pt tensor?

In [ ]:
n_fft = 2048
hop_length = 512
n_mels = 128
waveform_length = 661500

time_frames = math.floor(waveform_length / hop_length) + 1
pt_files = glob.glob(os.path.join(OUTPUT_DIR, '**', '*.pt'), recursive=True)
sample_tensor = torch.load(pt_files[0])

q3_output = {
    "time_frames": int(time_frames),
    "actual_tensor_shape": tuple(sample_tensor.shape)
}
q3_output

Q3 Answer: (2, 128, 1292)

---
## Build the CRNN Model
Model used for Q4 and Q5.

In [ ]:
class PrecomputedFeatureDataset(Dataset):
    def __init__(self, features_dir):
        self.files = glob.glob(os.path.join(features_dir, '**', '*.pt'), recursive=True)
        self.genres = sorted(['blues', 'classical', 'country', 'disco', 'hiphop',
                              'jazz', 'metal', 'pop', 'reggae', 'rock'])
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        genre = Path(file_path).parent.name
        label = self.genre_to_idx[genre]

        feature = torch.load(file_path)
        return feature, label

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(64 * 2, num_classes)

    def forward(self, x):
        x = self.cnn(x)

        b, c, f, t = x.shape
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(b, t, c * f)

        x, _ = self.lstm(x)
        x, _ = torch.max(x, dim=1)

        logits = self.fc(x)

        return logits

---
## Q4
What is the shape after second MaxPool2d(2), before permute()?

In [ ]:
seed_everything(42)
model = CRNN(num_classes=10)
dummy_input = torch.randn(32, 1, 128, 1292)
cnn_output = model.cnn(dummy_input)

q4_output = {
    "shape_after_second_maxpool": tuple(cnn_output.shape)
}
q4_output

Q4 Answer: (32, 64, 32, 323)

---
## Q5
How many trainable parameters are in the LSTM layer alone?

In [ ]:
lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
lstm_breakdown = []
for name, p in model.lstm.named_parameters():
    if p.requires_grad:
        lstm_breakdown.append((name, tuple(p.shape), int(p.numel())))

q5_output = {
    "lstm_trainable_parameters": int(lstm_params),
    "parameter_breakdown": lstm_breakdown
}
q5_output

Q5 Answer: 1082368

---
## Training
Train the model with the given setup.

In [ ]:
FEATURES_DIR = '/kaggle/working/features/train'

full_dataset = PrecomputedFeatureDataset(FEATURES_DIR)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

data_split_info = {
    "total_samples": int(len(full_dataset)),
    "train_samples": int(len(train_dataset)),
    "val_samples": int(len(val_dataset))
}
data_split_info

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_epochs = 10

training_setup = {
    "device": str(device),
    "model_parameters": int(sum(p.numel() for p in model.parameters())),
    "num_epochs": int(num_epochs)
}
training_setup

In [ ]:
history = []
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = 100 * train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    history.append({
        "epoch": int(epoch + 1),
        "train_loss": float(train_loss / len(train_loader)),
        "train_acc": float(train_acc),
        "val_loss": float(val_loss / len(val_loader)),
        "val_acc": float(val_acc)
    })

history

---
## Q6
Why is translation invariance along the Y-axis conceptually problematic for music?
Answer: Moving a pattern up the Y-axis changes pitch and can change instrument identity.

In [ ]:
q6_answer = "Moving a pattern up the Y-axis fundamentally changes its pitch and instrument identity (e.g., shifting a low bassline up 80 bins turns it into a high-pitched synth)."
q6_answer

Q6 Answer: Moving a pattern up the Y-axis changes pitch and can change instrument identity.

---
## Summary
- Q1: 500
- Q2: (2, 661500)
- Q3: (2, 128, 1292)
- Q4: (32, 64, 32, 323)
- Q5: 1082368
- Q6: Moving a pattern up the Y-axis changes pitch and can change instrument identity.